# 03. Annotate retinal cell types

This notebook starts from `data/interim/02_clustered.h5ad` and focuses on assigning biological cell-type labels to Leiden clusters.

Main outputs:
- marker gene UMAPs
- marker dotplots by cluster
- cluster-level marker score table
- editable manual annotation map
- annotated object saved to `data/processed/03_annotated.h5ad`
- annotation table saved to `results/annotation/03_cluster_annotations.csv`

## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
sns.set_style("whitegrid")

## Resolve project paths

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "samples.csv").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing config/samples.csv")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ANNOTATION_DIR = PROJECT_ROOT / "results" / "annotation"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

INPUT_H5AD = INTERIM_DIR / "02_clustered.h5ad"
OUTPUT_H5AD = PROCESSED_DIR / "03_annotated.h5ad"
ANNOTATION_CSV = ANNOTATION_DIR / "03_cluster_annotations.csv"
MARKER_SCORE_CSV = ANNOTATION_DIR / "03_cluster_marker_scores.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input object: {INPUT_H5AD}")

## Load clustered object

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)
adata

In [ ]:
CLUSTER_KEY = "leiden_res_0.50"

cluster_counts = adata.obs[CLUSTER_KEY].value_counts().sort_index()
cluster_counts

## Define retinal marker genes

In [ ]:
marker_genes = {
    "Rod": ["Rho", "Gnat1", "Pde6a", "Pde6b", "Cnga1", "Nr2e3", "Nrl", "Prph2", "Rom1"],
    "Cone": ["Pde6h", "Arr3", "Opn1mw", "Gnat2", "Pde6c", "Opn1sw"],
    "BC": ["Vsx1", "Vsx2", "Car10", "Otx2os1", "Gng13", "Nrxn3", "Gabrb3", "Trnp1", "Kcnma1", "Frmd3", "Gm4792", "Nyap2"],
    "AC": ["Tfap2a", "Tfap2b", "Pax6", "Frmd5", "Nrg3", "Elavl3", "Gad1", "Gad2"],
    "MG": ["Rlbp1", "Glul", "Dkk3", "Adamtsl1", "Gpr37", "Abca8a", "Rgs6", "Spc25", "Clu", "Apoe", "Aqp4"],
    "HC": ["Calb1", "Onecut1", "Slc4a3", "Onecut2", "Gm45459", "C1ql1"],
    "RGC": ["Nefl", "Stmn2", "Nrn1", "Pou4f1", "Pou4f2", "Sncg", "Stmn2", "Rbpms"],
    "RPE": ["Ttr", "Rdh5", "Rgr", "Slc16a8"],
    "Astrocyte": ["S100b", "Gfap", "Mlc1", "Prdx6", "Pdgfra"],
    "Microglia": ["Ctss", "C1qa", "Hexb", "Trem2"],
    "Endothelial": ["Cldn5", "Flt1", "Ly6c1", "Ptprb"],
}

available_marker_genes = {
    cell_type: [gene for gene in genes if gene in adata.var_names]
    for cell_type, genes in marker_genes.items()
}

missing_marker_genes = {
    cell_type: [gene for gene in genes if gene not in adata.var_names]
    for cell_type, genes in marker_genes.items()
}

available_marker_genes

In [ ]:
missing_marker_genes

## Inspect clusters and marker expression on UMAP

In [ ]:
sc.pl.umap(
    adata,
    color=["sample_id", "condition", CLUSTER_KEY],
    ncols=2,
    legend_loc="on data",
    size=2,
)

In [ ]:
key_markers = ["Rho", "Malat1", "Nrl", 'Rlbp1', 'Scgn', 'Arr3', 'Vsx2', 'Tfap2a', 'Elavl3', 'Prkca', 'Rbpms', "Chat", "Gad1", "Th", "Slc6a9"]
key_markers = [gene for gene in key_markers if gene in adata.var_names]

sc.pl.umap(
    adata,
    color=key_markers,
    ncols=4,
    wspace=0.5,
    size=2,
)

## Marker dotplot by Leiden cluster
question:
1.cluster id 6 both express MG and Astrocyte

In [ ]:
sc.pl.dotplot(
    adata,
    available_marker_genes,
    groupby=CLUSTER_KEY,
    standard_scale="var",
)

## Score marker programs per cell and summarize by cluster

These scores are a guide, not a replacement for reviewing the UMAP and dotplot. A cluster can score high for more than one marker set if it contains mixed cells, stressed cells, or doublets.

In [ ]:
score_columns = []

for cell_type, genes in available_marker_genes.items():
    if len(genes) < 2:
        continue
    score_col = f"score_{cell_type}"
    sc.tl.score_genes(adata, gene_list=genes, score_name=score_col)
    score_columns.append(score_col)

score_columns

In [ ]:
cluster_marker_scores = adata.obs.groupby(CLUSTER_KEY, observed=True)[score_columns].mean()
cluster_marker_scores["suggested_celltype"] = (
    cluster_marker_scores[score_columns]
    .idxmax(axis=1)
    .str.replace("score_", "", regex=False)
)
cluster_marker_scores["n_cells"] = cluster_counts.reindex(cluster_marker_scores.index).astype(int)

ordered_cols = ["n_cells", "suggested_celltype", *score_columns]
cluster_marker_scores = cluster_marker_scores[ordered_cols]
cluster_marker_scores.to_csv(MARKER_SCORE_CSV)
cluster_marker_scores

In [ ]:
plt.figure(figsize=(12, max(6, 0.35 * len(cluster_marker_scores))))
sns.heatmap(
    cluster_marker_scores[score_columns],
    cmap="vlag",
    center=0,
    linewidths=0.3,
    linecolor="white",
)
plt.title(f"Marker scores by {CLUSTER_KEY}")
plt.xlabel("Marker score")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

todo: cell cycle score

## Review top marker genes per cluster

This identifies genes enriched in each Leiden cluster. It helps confirm or correct the marker-score suggestions.

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby=CLUSTER_KEY,
    method="wilcoxon",
    pts=True,
)

top_cluster_markers = sc.get.rank_genes_groups_df(adata, group=None)
top_cluster_markers = top_cluster_markers.sort_values(["group", "pvals_adj", "scores"])
top_cluster_markers.to_csv(ANNOTATION_DIR / "03_rank_genes_by_cluster.csv", index=False)
top_cluster_markers.head(20)

In [ ]:
top_n = 8
cluster_top_genes = (
    top_cluster_markers
    .dropna(subset=["names"])
    .groupby("group", observed=True)
    .head(top_n)
    .groupby("group", observed=True)["names"]
    .apply(lambda genes: ", ".join(genes.astype(str)))
    .to_frame("top_marker_genes")
)

cluster_review = cluster_marker_scores.join(cluster_top_genes, how="left")
cluster_review.to_csv(ANNOTATION_DIR / "03_cluster_review_table.csv")
cluster_review

## Assign manual annotations

Edit `manual_annotation` after reviewing the UMAP, dotplot, marker scores, and top marker genes. The initial values are only suggestions from marker scoring.

Use consistent labels such as:

- `Rod`
- `Cone`
- `Muller_glia`
- `Bipolar`
- `Amacrine`
- `Horizontal`
- `RGC`
- `Microglia`
- `Endothelial`
- `RPE`
- `Astrocyte`
- `Unknown`

In [ ]:
manual_annotation = cluster_marker_scores["suggested_celltype"].to_dict()

# Review and edit this mapping before saving final annotations.
manual_annotation

In [ ]:
adata.obs["manual_celltype_annotation"] = adata.obs[CLUSTER_KEY].map(manual_annotation).astype("category")

annotation_table = pd.DataFrame({
    CLUSTER_KEY: list(manual_annotation.keys()),
    "manual_celltype_annotation": list(manual_annotation.values()),
})
annotation_table["n_cells"] = annotation_table[CLUSTER_KEY].map(cluster_counts).astype(int)
annotation_table = annotation_table.sort_values(CLUSTER_KEY, key=lambda s: s.astype(int))
annotation_table.to_csv(ANNOTATION_CSV, index=False)

annotation_table

## Visualize final annotations

In [ ]:
sc.pl.umap(
    adata,
    color=["manual_celltype_annotation", CLUSTER_KEY, "sample_id", "condition"],
    ncols=2,
    legend_loc="on data",
    size=2,
)

In [ ]:
celltype_counts = pd.crosstab(
    adata.obs["manual_celltype_annotation"],
    adata.obs["condition"],
)
celltype_counts

In [ ]:
celltype_props = pd.crosstab(
    adata.obs["condition"],
    adata.obs["manual_celltype_annotation"],
    normalize="index",
) * 100

ax = celltype_props.plot(kind="bar", stacked=True, figsize=(10, 6), colormap="tab20")
ax.set_ylabel("Percent of cells")
ax.set_xlabel("Condition")
ax.set_title("Cell type composition by condition")
ax.legend(title="Cell type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

celltype_props.round(2)

## Save annotated object

In [ ]:
adata.write(OUTPUT_H5AD)
print(f"Saved annotated AnnData: {OUTPUT_H5AD}")
print(f"Saved annotation table: {ANNOTATION_CSV}")
print(f"Saved marker score table: {MARKER_SCORE_CSV}")

## Next notebook

After reviewing and finalizing `manual_annotation`, the next notebook should run differential expression for `Insm1_KO` versus `Control` within the target cell types:

- `Rod`
- `Cone`
- `Muller_glia`

Because this project has one sample per condition, interpret DE as sample-level exploratory evidence rather than a replicated statistical experiment.